# Preparación y agregaciones — GHAW-H (Tarea 5)

Este notebook prepara los datos históricos del dataset **GHAW-H** para
estudiar cómo cambian los archivos Markdown de GitHub Agentic Workflows
(GH-AW) entre versiones, y genera las tablas agregadas que consume el
Notebook 2 (`02_boxplots_y_evolucion.ipynb`).

**Dataset**: [GHAW-H](https://huggingface.co/datasets/pavtch/GHAW-H)
(Valenzuela-Toledo, Kehrer y Panichella, 2026), versión de cita **v0.1.2**,
commit `9ccc840bd7e96fd90db3a16548d5482c2833c47d` del repositorio de Hugging
Face. Ver [`README.md`](README.md) de esta carpeta para cómo obtener los
datos.

De las 5 tablas publicadas usamos 4 (no se necesita `lock_file_snapshot`
para este análisis, que trabaja sobre el Markdown fuente):

| Tabla | Grano |
|---|---|
| `repository` | Un repositorio de GitHub |
| `source_markdown_file_history` | Una **historia**: el conjunto de versiones observadas de un mismo archivo |
| `source_markdown_file_version` | Una versión observada de un archivo, en un commit dado |
| `source_markdown_file_snapshot` | El contenido inmutable de una versión (ya separado en `frontmatter` y `body`) |


## 1. Carga y relación de las tablas

In [1]:
import numpy as np
import pandas as pd
import yaml
from pathlib import Path

pd.set_option("display.max_columns", 20)

RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

repository = pd.read_parquet(RAW_DIR / "repository.parquet")
history = pd.read_parquet(RAW_DIR / "source_markdown_file_history.parquet")
version = pd.read_parquet(RAW_DIR / "source_markdown_file_version.parquet")
snapshot = pd.read_parquet(RAW_DIR / "source_markdown_file_snapshot.parquet")

for name, df in [("repository", repository), ("history", history),
                  ("version", version), ("snapshot", snapshot)]:
    print(f"{name}: {df.shape[0]} filas x {df.shape[1]} columnas")

repository: 262 filas x 12 columnas
history: 604 filas x 2 columnas
version: 2820 filas x 8 columnas
snapshot: 2820 filas x 6 columnas


In [2]:
# Relacionar version -> snapshot (contenido) -> repository (dueño del archivo),
# y version -> history (cuántas versiones tiene la historia en total).
v = version.merge(
    snapshot[["source_markdown_file_snapshot_id", "repository_id", "path", "frontmatter", "body"]],
    on="source_markdown_file_snapshot_id", how="left", validate="many_to_one",
)
v = v.merge(
    repository[["repository_id", "repo_full_name"]],
    on="repository_id", how="left", validate="many_to_one",
)
v = v.merge(
    history[["source_markdown_file_history_id", "version_count"]],
    on="source_markdown_file_history_id", how="left", validate="many_to_one",
)

print(f"Tabla relacionada 'v': {v.shape[0]} filas x {v.shape[1]} columnas")
v.head(3)

Tabla relacionada 'v': 2820 filas x 14 columnas


,source_markdown_file_version_id,source_markdown_file_history_id,source_markdown_file_snapshot_id,commit_sha,committed_at,rank,predecessor_source_markdown_file_version_id,successor_source_markdown_file_version_id,repository_id,path,frontmatter,body,repo_full_name,version_count
0,1,1,1,957bfbb1cddd1ffac718a2d3f135ded4a85b73d1,2026-06-17T18:15:03Z,1,NaN,NaN,9089213639968762207,.github/workflows/daily-issue-triage.md,description: |\n Scheduled daily triage that ...,\n# Daily Issue Triage\n\n<!-- Note - this fil...,apache/cloudstack,1
1,2,2,2,a1bcae921367eb8c38d0e119124416f3c9fa75ff,2026-02-17T15:11:01Z,1,NaN,3.0,9089213639968762207,.github/workflows/daily-repo-status.md,description: |\n This workflow creates daily ...,\n# Daily Repo Status\n\nCreate an upbeat dail...,apache/cloudstack,4
2,3,2,3,c0db75b9fa6bdaff31297ee0ebe29d7ce1ef8459,2026-02-18T08:35:17Z,2,2.0,4.0,9089213639968762207,.github/workflows/daily-repo-status.md,description: |\n This workflow creates daily ...,\n# Daily Repo Status\n\nCreate an upbeat dail...,apache/cloudstack,4


Una **historia** (`source_markdown_file_history`) corresponde al
conjunto de versiones observadas de un mismo archivo a lo largo de distintos
commits. Una historia con una sola versión no tiene ninguna versión anterior
con la cual compararse, así que no puede aportar medidas de cambio; las
historias con dos o más versiones sí lo permiten y son la base de todo el
análisis que sigue.

In [3]:
n_repos = repository["repository_id"].nunique()
n_hist = history["source_markdown_file_history_id"].nunique()
n_ver = version["source_markdown_file_version_id"].nunique()
n_hist_1 = int((history["version_count"] == 1).sum())
n_hist_2plus = int((history["version_count"] >= 2).sum())

print(f"Repositorios:                {n_repos}")
print(f"Historias de archivo:        {n_hist}")
print(f"Versiones observadas:        {n_ver}")
print(f"  - historias con 1 versión: {n_hist_1}")
print(f"  - historias con >=2:       {n_hist_2plus} (estas se usan para calcular cambios)")
assert n_hist_1 + n_hist_2plus == n_hist

Repositorios:                262
Historias de archivo:        604
Versiones observadas:        2820
  - historias con 1 versión: 198
  - historias con >=2:       406 (estas se usan para calcular cambios)


## 2. Reconstrucción de las secuencias de versiones

Para poder comparar cada versión con la anterior, primero convertimos
`committed_at` a `datetime` y calculamos, para cada versión, las medidas por
versión (Sección 3) *antes* de encadenar predecesores — así el propio
auto-merge trae de una sola vez el `rank`, la fecha y las medidas de la
versión predecesora.

In [4]:
v["committed_at"] = pd.to_datetime(v["committed_at"], utc=True, errors="coerce")
print("Rango de fechas observado:", v["committed_at"].min(), "->", v["committed_at"].max())

Rango de fechas observado: 2025-10-03 20:05:08+00:00 -> 2026-06-27 01:41:44+00:00


## 3. Cálculo de medidas

**Longitud del body**: cantidad de palabras, usando separación por
espacios en blanco (`str.split()` sin argumentos, que colapsa cualquier
secuencia de espacios/tabs/saltos de línea). Distinguimos:
- `body` ausente (`None`) → longitud **ausente** (`NaN`).
- `body` presente pero vacío (`""`) → longitud **0** (contenido vacío, no
  ausente).

**Tamaño del frontmatter**: cantidad de claves de primer nivel del YAML
(sin contar claves anidadas dentro de ellas), parseando con `yaml.safe_load`.
Distinguimos cuatro casos, además del caso normal:
- `frontmatter` ausente (`None`) → tamaño **ausente**, categoría `missing`.
- `frontmatter` vacío (`""`) o que parsea a `None` → tamaño **0**, categoría
  `empty` (frontmatter presente pero sin contenido).
- Error al parsear el YAML → tamaño **ausente**, categoría `parse_error`.
- El YAML parsea pero no es un mapeo (p. ej. una lista) → tamaño
  **ausente**, categoría `not_a_mapping` (no tiene "claves de primer nivel").
- El YAML parsea a un `dict` → tamaño = cantidad de claves, categoría `ok`.

In [5]:
def count_words(text):
    """Cantidad de palabras separadas por espacios en blanco. None -> NaN, '' -> 0."""
    if text is None:
        return np.nan
    if text == "":
        return 0
    return len(text.split())


def frontmatter_keys(text):
    """Cantidad de claves de primer nivel del frontmatter YAML, y su categoría."""
    if text is None:
        return np.nan, "missing"
    if text.strip() == "":
        return 0, "empty"
    try:
        parsed = yaml.safe_load(text)
    except yaml.YAMLError:
        return np.nan, "parse_error"
    if parsed is None:
        return 0, "empty"
    if isinstance(parsed, dict):
        return len(parsed), "ok"
    return np.nan, "not_a_mapping"


v["body_length"] = v["body"].map(count_words)
_fm = v["frontmatter"].map(frontmatter_keys)
v["frontmatter_size"] = _fm.map(lambda t: t[0])
v["frontmatter_status"] = _fm.map(lambda t: t[1])

print("Longitud de body ausente:", int(v["body_length"].isna().sum()))
print()
print("Categorías de frontmatter observadas:")
print(v["frontmatter_status"].value_counts())

Longitud de body ausente: 0

Categorías de frontmatter observadas:
frontmatter_status
ok             2813
empty             6
parse_error       1
Name: count, dtype: int64


Guardamos la tabla de medidas por versión (`version_measures`), que es
una de las cuatro tablas que se persisten al final de este notebook.

In [6]:
version_measures = v[[
    "source_markdown_file_version_id", "source_markdown_file_history_id",
    "source_markdown_file_snapshot_id", "repository_id", "repo_full_name",
    "path", "rank", "committed_at", "body_length", "frontmatter_size", "frontmatter_status",
]].copy()
version_measures.head(3)

,source_markdown_file_version_id,source_markdown_file_history_id,source_markdown_file_snapshot_id,repository_id,repo_full_name,path,rank,committed_at,body_length,frontmatter_size,frontmatter_status
0,1,1,1,9089213639968762207,apache/cloudstack,.github/workflows/daily-issue-triage.md,1,2026-06-17 18:15:03+00:00,1020,9.0,ok
1,2,2,2,9089213639968762207,apache/cloudstack,.github/workflows/daily-repo-status.md,1,2026-02-17 15:11:01+00:00,105,7.0,ok
2,3,2,3,9089213639968762207,apache/cloudstack,.github/workflows/daily-repo-status.md,2,2026-02-18 08:35:17+00:00,105,7.0,ok


Ahora sí, reconstruimos la secuencia: para cada versión buscamos su
predecesora vía `predecessor_source_markdown_file_version_id` y traemos de
una vez su historia, `rank`, fecha y medidas. Una **transición** es la
comparación entre una versión y su predecesora observada; la primera
versión de cada historia (`rank == 1`, sin predecesor) nunca genera una
transición, por diseño.

Antes de aceptar una transición como válida, documentamos y filtramos los
casos problemáticos:
- el predecesor referenciado pertenece a **otra** historia (referencia
  inconsistente);
- el `rank` no es exactamente consecutivo (`rank` actual ≠ `rank` del
  predecesor + 1);
- falta la fecha (`committed_at`) de alguna de las dos versiones;
- el intervalo entre fechas es **negativo** (la versión "posterior" quedó
  registrada con una fecha anterior a su predecesora).

Cualquier transición que caiga en alguno de estos casos se excluye de
`transitions` (sus medidas de cambio quedarían sin sentido) y se informa el
conteo encontrado, aunque termine siendo cero.

In [7]:
pred_cols = ["source_markdown_file_version_id", "source_markdown_file_history_id",
             "rank", "committed_at", "body_length", "frontmatter_size"]
v = v.merge(
    v[pred_cols].add_prefix("pred_"),
    left_on="predecessor_source_markdown_file_version_id",
    right_on="pred_source_markdown_file_version_id",
    how="left",
)

has_pred = v["predecessor_source_markdown_file_version_id"].notna()
mismatch_history = has_pred & (v["source_markdown_file_history_id"] != v["pred_source_markdown_file_history_id"])
rank_gap = v["rank"] - v["pred_rank"]
non_consecutive_rank = has_pred & (rank_gap != 1)
missing_date = has_pred & (v["committed_at"].isna() | v["pred_committed_at"].isna())
interval_days = (v["committed_at"] - v["pred_committed_at"]).dt.total_seconds() / 86400
negative_interval = has_pred & (interval_days < 0)

print(f"Candidatas a transición (tienen predecesor): {int(has_pred.sum())}")
print(f"  - predecesor de otra historia:  {int(mismatch_history.sum())}")
print(f"  - rank no consecutivo:          {int(non_consecutive_rank.sum())}")
print(f"  - fecha ausente en el par:      {int(missing_date.sum())}")
print(f"  - intervalo negativo:           {int(negative_interval.sum())}")

invalid_transition = mismatch_history | non_consecutive_rank | missing_date | negative_interval
transitions_base = v.loc[has_pred & ~invalid_transition].copy()
print(f"Transiciones válidas retenidas: {len(transitions_base)}")

Candidatas a transición (tienen predecesor): 2216
  - predecesor de otra historia:  0
  - rank no consecutivo:          0
  - fecha ausente en el par:      0
  - intervalo negativo:           0
Transiciones válidas retenidas: 2216


En este dataset las cuatro validaciones dan 0 casos: la secuencia de
versiones es internamente consistente (rank siempre +1, sin referencias
cruzadas entre historias, sin fechas faltantes ni intervalos negativos). De
todas formas la lógica de arriba queda para detectar y excluir
automáticamente estos casos si aparecieran al regenerar el dataset con una
versión futura de GHAW-H.

Con las transiciones ya filtradas, calculamos las medidas de cambio:

| Medida | Cálculo |
|---|---|
| Cambio de longitud del body | `body_length` actual − `body_length` del predecesor |
| Magnitud del cambio de longitud | Valor absoluto del cambio anterior |
| Cambio del tamaño del frontmatter | `frontmatter_size` actual − `frontmatter_size` del predecesor |
| Tiempo entre versiones | Diferencia en días (fraccionarios) entre `committed_at` de ambas versiones |

Usamos días **fraccionarios** (`total_seconds() / 86400`) en vez de días
enteros porque muchas transiciones ocurren dentro del mismo día calendario
(commits seguidos) y truncar a enteros las colapsaría todas a 0, perdiendo
la variación real que después resumimos con la mediana.

Un `delta_body_length` igual a 0 **no** implica que el contenido sea
idéntico: puede haber sustituciones de palabras por otras de igual cantidad,
o cambios de valores en el frontmatter que no alteran la cantidad de
palabras del body.

In [8]:
transitions_base["delta_body_length"] = transitions_base["body_length"] - transitions_base["pred_body_length"]
transitions_base["abs_delta_body_length"] = transitions_base["delta_body_length"].abs()
transitions_base["delta_frontmatter_size"] = transitions_base["frontmatter_size"] - transitions_base["pred_frontmatter_size"]
transitions_base["days_between_versions"] = (
    transitions_base["committed_at"] - transitions_base["pred_committed_at"]
).dt.total_seconds() / 86400

for col in ["delta_body_length", "delta_frontmatter_size", "days_between_versions"]:
    print(f"{col}: {int(transitions_base[col].isna().sum())} valores ausentes de {len(transitions_base)}")

print()
print("Transiciones con cambio de tamaño de frontmatter (!= 0):",
      int((transitions_base["delta_frontmatter_size"] != 0).sum()))

delta_body_length: 0 valores ausentes de 2216
delta_frontmatter_size: 2 valores ausentes de 2216
days_between_versions: 0 valores ausentes de 2216

Transiciones con cambio de tamaño de frontmatter (!= 0): 372


`delta_frontmatter_size` sale ausente en los casos donde el frontmatter
de la versión actual o la predecesora no pudo interpretarse como cantidad de
claves (`missing`, `parse_error` o `not_a_mapping`) — coherente con lo
documentado arriba: no se inventa un valor, se conserva como ausente y ya
quedó contado.

Construimos `transitions`, la segunda tabla a persistir, manteniendo los
identificadores de las dos versiones comparadas en cada fila.

In [9]:
transitions = transitions_base[[
    "source_markdown_file_version_id", "predecessor_source_markdown_file_version_id",
    "source_markdown_file_history_id", "repository_id", "repo_full_name", "path",
    "committed_at", "pred_committed_at",
    "body_length", "pred_body_length", "delta_body_length", "abs_delta_body_length",
    "frontmatter_size", "pred_frontmatter_size", "delta_frontmatter_size",
    "days_between_versions",
]].rename(columns={
    "source_markdown_file_version_id": "version_id",
    "predecessor_source_markdown_file_version_id": "predecessor_version_id",
    "source_markdown_file_history_id": "history_id",
    "pred_committed_at": "predecessor_committed_at",
    "pred_body_length": "predecessor_body_length",
    "pred_frontmatter_size": "predecessor_frontmatter_size",
}).reset_index(drop=True)

print(f"transitions: {transitions.shape[0]} filas x {transitions.shape[1]} columnas")
transitions.head(3)

transitions: 2216 filas x 16 columnas


,version_id,predecessor_version_id,history_id,repository_id,repo_full_name,path,committed_at,predecessor_committed_at,body_length,predecessor_body_length,delta_body_length,abs_delta_body_length,frontmatter_size,predecessor_frontmatter_size,delta_frontmatter_size,days_between_versions
0,3,2.0,2,9089213639968762207,apache/cloudstack,.github/workflows/daily-repo-status.md,2026-02-18 08:35:17+00:00,2026-02-17 15:11:01+00:00,105,105.0,0.0,0.0,7.0,7.0,0.0,0.725185
1,4,3.0,2,9089213639968762207,apache/cloudstack,.github/workflows/daily-repo-status.md,2026-05-29 09:15:49+00:00,2026-02-18 08:35:17+00:00,104,105.0,-1.0,1.0,7.0,7.0,0.0,100.028148
2,5,4.0,2,9089213639968762207,apache/cloudstack,.github/workflows/daily-repo-status.md,2026-06-17 18:15:03+00:00,2026-05-29 09:15:49+00:00,104,104.0,0.0,0.0,8.0,7.0,1.0,19.374468


## 4. Agregaciones por archivo y por mes

**Resumen por archivo** (`file_summary`, una fila por historia):
cantidad de versiones, primera y última fecha observada, longitud inicial
(`rank == 1`) y final (`rank` máximo) del body, el **cambio neto**
(longitud de la última versión menos la de la primera) y la mediana del
tiempo entre versiones de esa historia. Incluye **todas** las historias
(también las de una sola versión, con `median_days_between_versions`
ausente por no tener ninguna transición).

In [10]:
vm = version_measures.rename(columns={"source_markdown_file_history_id": "history_id"})

file_summary = (
    vm.sort_values(["history_id", "rank"])
    .groupby("history_id")
    .agg(
        version_count=("rank", "size"),
        first_date=("committed_at", "min"),
        last_date=("committed_at", "max"),
        first_length=("body_length", "first"),
        last_length=("body_length", "last"),
    )
    .reset_index()
)
file_summary["net_length_change"] = file_summary["last_length"] - file_summary["first_length"]

median_days = (
    transitions.groupby("history_id")["days_between_versions"]
    .median()
    .rename("median_days_between_versions")
)
file_summary = file_summary.merge(median_days, on="history_id", how="left")

print(f"file_summary: {file_summary.shape[0]} filas x {file_summary.shape[1]} columnas")
print("historias sin mediana de tiempo entre versiones (1 sola versión):",
      int(file_summary["median_days_between_versions"].isna().sum()))
file_summary.head(3)

file_summary: 604 filas x 8 columnas
historias sin mediana de tiempo entre versiones (1 sola versión): 198


,history_id,version_count,first_date,last_date,first_length,last_length,net_length_change,median_days_between_versions
0,1,1,2026-06-17 18:15:03+00:00,2026-06-17 18:15:03+00:00,1020,1020,0,NaN
1,2,4,2026-02-17 15:11:01+00:00,2026-06-17 18:15:03+00:00,105,104,-1,19.374468
2,3,3,2026-02-09 22:57:17+00:00,2026-02-09 23:27:44+00:00,52,51,-1,0.010573


**Resumen por archivo y mes** (`file_month_summary`, una fila por
historia con transiciones observadas durante un mes dado): cantidad de
transiciones y mediana de la magnitud del cambio de longitud del body. Cada
transición se asigna al **mes de la versión posterior** (la más reciente de
las dos comparadas). No completamos artificialmente los meses sin
transiciones con un valor cero: simplemente no aparece esa combinación
historia-mes en la tabla, que es el comportamiento por defecto de
`groupby` (no reindexamos contra el producto cartesiano historia×mes).

In [11]:
transitions["month"] = transitions["committed_at"].dt.tz_localize(None).dt.to_period("M")

file_month_summary = (
    transitions.groupby(["history_id", "month"])
    .agg(
        n_transitions=("version_id", "size"),
        median_abs_delta_body_length=("abs_delta_body_length", "median"),
    )
    .reset_index()
)
file_month_summary["month"] = file_month_summary["month"].astype(str)

print(f"file_month_summary: {file_month_summary.shape[0]} filas x {file_month_summary.shape[1]} columnas")
print("Meses cubiertos:", sorted(file_month_summary["month"].unique()))
file_month_summary.head(3)

file_month_summary: 768 filas x 4 columnas
Meses cubiertos: ['2025-10', '2025-11', '2025-12', '2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06']


,history_id,month,n_transitions,median_abs_delta_body_length
0,2,2026-02,1,0.0
1,2,2026-05,1,1.0
2,2,2026-06,1,0.0


### Guardar las tablas para el Notebook 2

In [12]:
transitions_to_save = transitions.drop(columns=["month"])

version_measures.to_parquet(PROCESSED_DIR / "version_measures.parquet", index=False)
transitions_to_save.to_parquet(PROCESSED_DIR / "transitions.parquet", index=False)
file_summary.to_parquet(PROCESSED_DIR / "file_summary.parquet", index=False)
file_month_summary.to_parquet(PROCESSED_DIR / "file_month_summary.parquet", index=False)

for name, df in [("version_measures", version_measures), ("transitions", transitions_to_save),
                  ("file_summary", file_summary), ("file_month_summary", file_month_summary)]:
    print(f"{name}.parquet: {df.shape[0]} filas x {df.shape[1]} columnas")

version_measures.parquet: 2820 filas x 11 columnas
transitions.parquet: 2216 filas x 16 columnas
file_summary.parquet: 604 filas x 8 columnas
file_month_summary.parquet: 768 filas x 4 columnas


Con estas 4 tablas guardadas en `data/processed/`, el Notebook 2
(`02_boxplots_y_evolucion.ipynb`) puede construir los boxplots y la
inspección de casos sin depender de ninguna variable en memoria de este
notebook.